will start by cleaning up the ACRG dataset, which i will use as a training set

In [15]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as pls

horizon_months = 24
out_dir = Path("data")
out_dir.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 160)

acrg = pd.read_csv("acrg_clinical_full.csv")
need = ["patient", "sex", "age", "t_stage", "m_stage", "positive_nodes", "who_simple", "recurrence_event", "dfs_months", "outcome_inconsistent"]

assert not [c for c in need if c not in acrg.columns], "there are missing columns in the ACRG csv"
assert acrg["patient"].is_unique
print(acrg.shape)

(300, 60)


In [16]:
df = pd.DataFrame({
    "patient_id": acrg["patient"].astype(str),
    "age": acrg["age"],
    "sex": acrg["sex"].map({"M": "Male", "F": "Female"}),
    "serosa":(acrg["t_stage"] >= 3).astype(float).where(acrg["t_stage"].notna()),
    "n_level": pd.cut(acrg["positive_nodes"], [-1, 0, 6, np.inf], labels=[0, 1, 2]).astype(float),
    "grade_g3": acrg["who_simple"].map({"p/d": 1.0, "signet": 1.0, "w/d~m/d": 0.0}),
    "m1": (acrg["m_stage"] == 1).astype(float).where(acrg["m_stage"].notna()),
    "event": acrg["recurrence_event"],
    "time_months": acrg["dfs_months"],
    "location": acrg["location_code"].map({"antrum": "antrum", "body": "body", "cardia": "cardia", "whole": "other"}),
    "outcome_inconsistent": acrg["outcome_inconsistent"].astype(bool),
})
df["outcome_known"] = df["event"].notna() & df["time_months"].notna()
df.head()

,patient_id,age,sex,serosa,n_level,grade_g3,m1,event,time_months,location,outcome_inconsistent,outcome_known
0,107,67,Male,0.0,1.0,0.0,0.0,NaN,3.966667,body,False,False
1,108,68,Female,0.0,1.0,1.0,0.0,NaN,4.033333,body,False,False
2,109,42,Female,0.0,1.0,1.0,0.0,0.0,74.966667,antrum,False,True
3,160,69,Male,0.0,1.0,1.0,0.0,0.0,89.766667,antrum,False,True
4,164,68,Male,1.0,2.0,1.0,0.0,0.0,84.600000,antrum,False,True


In [17]:
core = ["age", "sex", "serosa", "n_level", "event", "time_months"]

steps = [
    ("all patients", pd.Series(True, index = df.index)),
    ("outcome is known", df["outcome_known"]),
    ("no distant metastasis", df["m1"].eq(0)),
    ("core variables are complete", df[core].notna().all(axis = 1)),
    ("no contradictory outcomes are recorded", ~df["outcome_inconsistent"]),
    ("follow-up time > 0", df["time_months"] > 0),
]

mask, rows = pd.Series(True, index = df.index), []
for label, m in steps:
    mask = mask & m.fillna(False)
    rows.append((label, int(mask.sum()), int(df.loc[mask, "event"].sum())))

flow = pd.DataFrame(rows, columns = ["step", "patients", "events"])
df["main_analysis"] = mask
flow

,step,patients,events
0,all patients,300,125
1,outcome is known,282,125
2,no distant metastasis,257,105
3,core variables are complete,255,105
4,no contradictory outcomes are recorded,252,103
5,follow-up time > 0,252,103


In [18]:
out_cols = ["patient_id", "age", "sex", "serosa", "n_level", "grade_g3", "event", "time_months", "location"]
final = df.loc[df["main_analysis"], out_cols].reset_index(drop = True)

print("Percent missing in the final set:"); display((final.isna().mean() * 100).round(1))
problems = []
if not final["age"].between(18, 100).all():   problems.append("age outside 18-100")
if not (final["time_months"] > 0).all():      problems.append("non-positive time")
if not final["event"].isin([0, 1]).all():     problems.append("event not 0/1")
if not final["patient_id"].is_unique:         problems.append("duplicate ids")
print("Checks:", "all passed" if not problems else problems)

Percent missing in the final set:


patient_id      0.0
age             0.0
sex             0.0
serosa          0.0
n_level         0.0
grade_g3       12.7
event           0.0
time_months     0.0
location        0.0
dtype: float64

Checks: all passed


In [19]:
final.to_csv(out_dir / "acrg_clean.csv", index=False)
flow.to_csv(out_dir / "acrg_flowchart.csv", index=False)
print("saved", final.shape)

saved (252, 9)
